# 01 — Audit exploratoire des résultats de votations OFS

Exploration visuelle du dataset produit par `python scripts/build_dataset.py`.
**Rien ici n'est nécessaire au pipeline**: l'audit officiel est `reports/data_audit.md`, généré par `scripts/audit_votes.py`.
Chaque cellule ne dépend que de la cellule de chargement.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pytest.ini").exists())
votes = pd.read_parquet(ROOT / "data" / "processed" / "votes_communes.parquet")
# Vraies communes et vraies questions d'approbation (hors unités spéciales et questions subsidiaires)
communes = votes[~votes["is_special_unit"] & ~votes["is_tiebreak_question"]]
COLOR = "#2a6fdb"
votes.shape, communes.shape

## Aperçu

In [ ]:
votes.head()

In [ ]:
pd.Series({
    "lignes": len(votes),
    "objets": votes["vote_id"].nunique(),
    "dates": votes["vote_date"].nunique(),
    "communes (hors spéciales)": votes.loc[~votes["is_special_unit"], "municipality_id"].nunique(),
    "unités spéciales": votes.loc[votes["is_special_unit"], "municipality_id"].nunique(),
    "première date": votes["vote_date"].min().date(),
    "dernière date": votes["vote_date"].max().date(),
})

## Distribution de `yes_share`

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3.5))
ax.hist(communes["yes_share"], bins=100, color=COLOR)
ax.set(xlabel="yes_share", ylabel="lignes commune × objet", title="Part de oui par commune et objet")
ax.spines[["top", "right"]].set_visible(False)

## Couverture: communes avec résultat par objet

In [ ]:
per_vote = communes.groupby(["vote_date", "vote_id"], observed=True)["municipality_id"].nunique().reset_index()
fig, ax = plt.subplots(figsize=(8, 3.5))
ax.plot(per_vote["vote_date"], per_vote["municipality_id"], ".", color=COLOR, markersize=4)
ax.set(ylabel="communes avec résultat", title="Saut en 1981: données 1960-1980 incomplètes (AG, FR, TI manquants)")
ax.spines[["top", "right"]].set_visible(False)

## Participation

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3.5))
yearly = communes.groupby(communes["vote_date"].dt.year)["turnout"].median()
ax.plot(yearly.index, yearly.values, color=COLOR, linewidth=2)
ax.set(ylabel="participation médiane", title="Participation communale médiane par année")
ax.spines[["top", "right"]].set_visible(False)
communes["turnout"].isna().groupby(communes["vote_date"].dt.year).sum().loc[lambda s: s > 0]

## Anomalies signalées (conservées, non corrigées)

In [ ]:
flags = ["flag_valid_gt_ballots", "flag_yes_no_gt_valid"]
votes.loc[votes[flags].any(axis=1), ["vote_date", "municipality_name", "canton", "ballots_cast",
                                     "valid_ballots", "votes_yes", "votes_no", *flags]].head(20)

## Unités spéciales (Suisses de l'étranger, etc.)

In [ ]:
votes[votes["is_special_unit"]].groupby("municipality_name", observed=True)["vote_date"].agg(["count", "min", "max"])